In [1]:
# ============================================================
#   Simple Movie Recommendation System
#   Concepts: Collaborative Filtering + Cosine Similarity
#   Tools: Python, Pandas, NumPy
# ============================================================

import numpy as np
import pandas as pd

# ─────────────────────────────────────────────
# STEP 1: Create a Sample Dataset
# Rows = Users, Columns = Movies
# Ratings are from 1 (bad) to 5 (great), 0 = not watched
# ─────────────────────────────────────────────

data = {
    'Inception':      [5, 4, 0, 0, 1, 0],
    'Interstellar':   [4, 5, 0, 0, 2, 0],
    'The Matrix':     [3, 0, 5, 4, 0, 0],
    'John Wick':      [0, 0, 4, 5, 0, 3],
    'Titanic':        [0, 0, 0, 0, 5, 4],
    'The Notebook':   [0, 0, 0, 0, 4, 5],
    'Avengers':       [4, 3, 5, 0, 0, 0],
    'Toy Story':      [0, 0, 0, 3, 5, 4],
}

users = ['Alice', 'Bob', 'Charlie', 'David', 'Eve', 'Frank']

# Create the user-item ratings matrix
ratings_df = pd.DataFrame(data, index=users)

print("=" * 55)
print("       USER - MOVIE RATINGS MATRIX")
print("=" * 55)
print(ratings_df)
print()


# ─────────────────────────────────────────────
# STEP 2: Compute Cosine Similarity Between Users
#
# Cosine similarity measures the angle between two vectors.
# A score of 1.0 = identical taste, 0.0 = completely different.
#
# Formula:
#   similarity(A, B) = (A · B) / (||A|| * ||B||)
# ─────────────────────────────────────────────

def cosine_similarity(vec1, vec2):
    """Compute cosine similarity between two user rating vectors."""
    dot_product = np.dot(vec1, vec2)             # A · B
    norm1 = np.linalg.norm(vec1)                 # ||A||
    norm2 = np.linalg.norm(vec2)                 # ||B||

    if norm1 == 0 or norm2 == 0:
        return 0.0  # Avoid division by zero

    return dot_product / (norm1 * norm2)


def compute_user_similarity(ratings):
    """Build a full similarity matrix for all users."""
    n_users = len(ratings)
    similarity_matrix = np.zeros((n_users, n_users))

    for i in range(n_users):
        for j in range(n_users):
            similarity_matrix[i][j] = cosine_similarity(
                ratings.iloc[i].values,
                ratings.iloc[j].values
            )

    return pd.DataFrame(
        similarity_matrix,
        index=ratings.index,
        columns=ratings.index
    )


similarity_df = compute_user_similarity(ratings_df)

print("=" * 55)
print("       USER SIMILARITY MATRIX (Cosine)")
print("=" * 55)
print(similarity_df.round(2))
print()


# ─────────────────────────────────────────────
# STEP 3: Generate Recommendations for a Target User
#
# Logic:
#  1. Find the most similar users to the target user
#  2. Look at movies they liked that the target hasn't watched
#  3. Score each unwatched movie using a weighted average:
#       score = sum(similarity * rating) / sum(similarity)
# ─────────────────────────────────────────────

def get_recommendations(target_user, ratings, similarity, top_n=3):
    """
    Recommend movies for target_user based on similar users' ratings.

    Parameters:
        target_user (str): The user to recommend movies for
        ratings (DataFrame): The full user-item ratings matrix
        similarity (DataFrame): The user similarity matrix
        top_n (int): Number of top recommendations to return
    """

    # Movies the target user has NOT watched yet (rating == 0)
    target_ratings = ratings.loc[target_user]
    unwatched_movies = target_ratings[target_ratings == 0].index.tolist()

    if not unwatched_movies:
        print(f"{target_user} has watched everything!")
        return

    scores = {}

    for movie in unwatched_movies:
        weighted_sum = 0
        similarity_sum = 0

        for other_user in ratings.index:
            if other_user == target_user:
                continue

            sim = similarity.loc[target_user, other_user]
            rating = ratings.loc[other_user, movie]

            # Only consider users who have actually watched this movie
            if rating > 0 and sim > 0:
                weighted_sum += sim * rating
                similarity_sum += sim

        # Compute the weighted average score
        if similarity_sum > 0:
            scores[movie] = weighted_sum / similarity_sum

    if not scores:
        print(f"No recommendations found for {target_user}.")
        return

    # Sort movies by predicted score (highest first)
    sorted_recommendations = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    print("=" * 55)
    print(f"  TOP {top_n} RECOMMENDATIONS FOR: {target_user.upper()}")
    print("=" * 55)
    for rank, (movie, score) in enumerate(sorted_recommendations[:top_n], start=1):
        print(f"  {rank}. {movie:<20} Predicted Score: {score:.2f}")
    print()


# ─────────────────────────────────────────────
# STEP 4: Run Recommendations
# ─────────────────────────────────────────────

# Test recommendations for different users
get_recommendations('Alice', ratings_df, similarity_df, top_n=3)
get_recommendations('Charlie', ratings_df, similarity_df, top_n=3)
get_recommendations('Eve', ratings_df, similarity_df, top_n=3)


# ─────────────────────────────────────────────
# STEP 5: Interactive Mode — Recommend for Any User
# ─────────────────────────────────────────────

print("=" * 55)
print("        INTERACTIVE RECOMMENDATION MODE")
print("=" * 55)
print(f"Available users: {', '.join(users)}")
user_input = input("Enter a username to get recommendations: ").strip()

if user_input in users:
    get_recommendations(user_input, ratings_df, similarity_df, top_n=3)
else:
    print(f"User '{user_input}' not found. Please choose from: {', '.join(users)}")

       USER - MOVIE RATINGS MATRIX
         Inception  Interstellar  The Matrix  John Wick  Titanic  \
Alice            5             4           3          0        0   
Bob              4             5           0          0        0   
Charlie          0             0           5          4        0   
David            0             0           4          5        0   
Eve              1             2           0          0        5   
Frank            0             0           0          3        4   

         The Notebook  Avengers  Toy Story  
Alice               0         4          0  
Bob                 0         3          0  
Charlie             0         5          0  
David               0         0          3  
Eve                 4         0          5  
Frank               5         0          4  

       USER SIMILARITY MATRIX (Cosine)
         Alice   Bob  Charlie  David   Eve  Frank
Alice     1.00  0.91     0.53   0.21  0.19   0.00
Bob       0.91  1.00     0.26   0

In [1]:
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0"/>
  <title>Series Recommender</title>
  <link href="https://fonts.googleapis.com/css2?family=Bebas+Neue&family=DM+Sans:wght@300;400;500;600&display=swap" rel="stylesheet"/>
  <style>
    *, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }

    :root {
      --bg:        #0a0a0f;
      --surface:   #13131c;
      --card:      #1a1a28;
      --border:    #2a2a40;
      --accent:    #e63946;
      --gold:      #f4a261;
      --text:      #e8e8f0;
      --muted:     #7070a0;
      --success:   #2ec4b6;
    }

    body {
      background: var(--bg);
      color: var(--text);
      font-family: 'DM Sans', sans-serif;
      min-height: 100vh;
      overflow-x: hidden;
    }

    /* ── background grain ── */
    body::before {
      content: '';
      position: fixed; inset: 0;
      background-image: url("data:image/svg+xml,%3Csvg viewBox='0 0 200 200' xmlns='http://www.w3.org/2000/svg'%3E%3Cfilter id='n'%3E%3CfeTurbulence type='fractalNoise' baseFrequency='0.9' numOctaves='4' stitchTiles='stitch'/%3E%3C/filter%3E%3Crect width='100%25' height='100%25' filter='url(%23n)' opacity='0.04'/%3E%3C/svg%3E");
      pointer-events: none; z-index: 0;
    }

    /* ── hero header ── */
    header {
      position: relative;
      padding: 60px 40px 40px;
      text-align: center;
      border-bottom: 1px solid var(--border);
      overflow: hidden;
    }
    header::after {
      content: '';
      position: absolute;
      bottom: -1px; left: 50%; transform: translateX(-50%);
      width: 200px; height: 2px;
      background: var(--accent);
    }
    .hero-label {
      font-size: 11px;
      letter-spacing: 4px;
      text-transform: uppercase;
      color: var(--accent);
      margin-bottom: 12px;
    }
    h1 {
      font-family: 'Bebas Neue', sans-serif;
      font-size: clamp(48px, 8vw, 96px);
      letter-spacing: 4px;
      line-height: 1;
      background: linear-gradient(135deg, #fff 40%, var(--gold));
      -webkit-background-clip: text;
      -webkit-text-fill-color: transparent;
      background-clip: text;
    }
    .subtitle {
      margin-top: 14px;
      color: var(--muted);
      font-size: 14px;
      font-weight: 300;
      letter-spacing: 1px;
    }

    /* ── main layout ── */
    main {
      position: relative; z-index: 1;
      max-width: 1100px;
      margin: 0 auto;
      padding: 50px 24px 80px;
    }

    /* ── section titles ── */
    .section-title {
      font-family: 'Bebas Neue', sans-serif;
      font-size: 22px;
      letter-spacing: 3px;
      color: var(--gold);
      margin-bottom: 20px;
      display: flex;
      align-items: center;
      gap: 12px;
    }
    .section-title::after {
      content: '';
      flex: 1;
      height: 1px;
      background: var(--border);
    }

    /* ── user selector ── */
    .user-grid {
      display: flex;
      flex-wrap: wrap;
      gap: 12px;
      margin-bottom: 40px;
    }
    .user-btn {
      padding: 10px 24px;
      border-radius: 4px;
      border: 1px solid var(--border);
      background: var(--card);
      color: var(--muted);
      font-family: 'DM Sans', sans-serif;
      font-size: 14px;
      font-weight: 500;
      cursor: pointer;
      transition: all 0.2s ease;
      letter-spacing: 0.5px;
    }
    .user-btn:hover {
      border-color: var(--accent);
      color: var(--text);
      transform: translateY(-2px);
    }
    .user-btn.active {
      background: var(--accent);
      border-color: var(--accent);
      color: #fff;
      transform: translateY(-2px);
      box-shadow: 0 6px 20px rgba(230,57,70,0.35);
    }

    /* ── recommend button ── */
    #recommend-btn {
      display: block;
      width: 100%;
      max-width: 340px;
      margin: 0 auto 50px;
      padding: 16px;
      background: var(--accent);
      color: #fff;
      font-family: 'Bebas Neue', sans-serif;
      font-size: 20px;
      letter-spacing: 3px;
      border: none;
      border-radius: 4px;
      cursor: pointer;
      transition: all 0.25s ease;
      box-shadow: 0 4px 20px rgba(230,57,70,0.3);
    }
    #recommend-btn:hover {
      background: #c1121f;
      transform: translateY(-2px);
      box-shadow: 0 8px 28px rgba(230,57,70,0.45);
    }
    #recommend-btn:disabled {
      background: var(--border);
      color: var(--muted);
      cursor: not-allowed;
      transform: none;
      box-shadow: none;
    }

    /* ── results area ── */
    #results { animation: fadeUp 0.4s ease; }
    @keyframes fadeUp {
      from { opacity: 0; transform: translateY(20px); }
      to   { opacity: 1; transform: translateY(0); }
    }

    .result-header {
      text-align: center;
      margin-bottom: 30px;
    }
    .result-header p {
      color: var(--muted);
      font-size: 13px;
      letter-spacing: 1px;
      margin-top: 6px;
    }
    .result-name {
      font-family: 'Bebas Neue', sans-serif;
      font-size: 40px;
      letter-spacing: 4px;
      color: var(--accent);
    }

    /* ── recommendation cards ── */
    .cards-grid {
      display: grid;
      grid-template-columns: repeat(auto-fit, minmax(280px, 1fr));
      gap: 20px;
      margin-bottom: 50px;
    }
    .rec-card {
      background: var(--card);
      border: 1px solid var(--border);
      border-radius: 8px;
      padding: 28px 24px;
      position: relative;
      overflow: hidden;
      transition: transform 0.2s ease, border-color 0.2s ease;
    }
    .rec-card:hover {
      transform: translateY(-4px);
      border-color: var(--accent);
    }
    .rec-card::before {
      content: '';
      position: absolute;
      top: 0; left: 0; right: 0;
      height: 3px;
      background: linear-gradient(90deg, var(--accent), var(--gold));
    }
    .rank-badge {
      font-family: 'Bebas Neue', sans-serif;
      font-size: 48px;
      color: var(--border);
      line-height: 1;
      margin-bottom: 8px;
    }
    .series-name {
      font-size: 18px;
      font-weight: 600;
      color: var(--text);
      margin-bottom: 14px;
      line-height: 1.3;
    }
    .score-bar-wrap {
      margin-top: 4px;
    }
    .score-label {
      display: flex;
      justify-content: space-between;
      font-size: 11px;
      color: var(--muted);
      letter-spacing: 1px;
      text-transform: uppercase;
      margin-bottom: 6px;
    }
    .score-value { color: var(--gold); font-weight: 600; }
    .score-bar {
      height: 4px;
      background: var(--border);
      border-radius: 2px;
      overflow: hidden;
    }
    .score-fill {
      height: 100%;
      background: linear-gradient(90deg, var(--accent), var(--gold));
      border-radius: 2px;
      transition: width 0.8s cubic-bezier(0.4,0,0.2,1);
      width: 0%;
    }

    /* ── ratings table ── */
    .table-wrap {
      overflow-x: auto;
      border: 1px solid var(--border);
      border-radius: 8px;
      margin-bottom: 50px;
    }
    table {
      width: 100%;
      border-collapse: collapse;
      font-size: 13px;
    }
    thead th {
      background: var(--surface);
      padding: 14px 16px;
      text-align: center;
      font-family: 'Bebas Neue', sans-serif;
      font-size: 13px;
      letter-spacing: 2px;
      color: var(--gold);
      border-bottom: 1px solid var(--border);
      white-space: nowrap;
    }
    thead th:first-child { text-align: left; }
    tbody tr { border-bottom: 1px solid var(--border); transition: background 0.15s; }
    tbody tr:last-child { border-bottom: none; }
    tbody tr:hover { background: rgba(255,255,255,0.03); }
    tbody td {
      padding: 12px 16px;
      text-align: center;
      color: var(--muted);
    }
    tbody td:first-child {
      text-align: left;
      font-weight: 500;
      color: var(--text);
    }
    .rating-cell {
      width: 32px; height: 32px;
      border-radius: 4px;
      display: inline-flex;
      align-items: center;
      justify-content: center;
      font-weight: 600;
      font-size: 13px;
    }
    .r5 { background: rgba(230,57,70,0.25); color: var(--accent); }
    .r4 { background: rgba(244,162,97,0.2); color: var(--gold); }
    .r3 { background: rgba(46,196,182,0.15); color: var(--success); }
    .r2 { background: rgba(255,255,255,0.07); color: var(--muted); }
    .r1 { background: rgba(255,255,255,0.04); color: #444; }
    .r0 { color: #333; font-size: 11px; }

    /* ── similarity table ── */
    .sim-cell {
      font-size: 12px;
      font-weight: 500;
    }
    .sim-high { color: var(--accent); }
    .sim-mid  { color: var(--gold); }
    .sim-low  { color: var(--muted); }

    /* ── no selection notice ── */
    .notice {
      text-align: center;
      padding: 60px 20px;
      color: var(--muted);
      font-size: 14px;
      letter-spacing: 1px;
    }
    .notice span {
      display: block;
      font-family: 'Bebas Neue', sans-serif;
      font-size: 48px;
      color: var(--border);
      margin-bottom: 12px;
    }

    /* ── footer ── */
    footer {
      text-align: center;
      padding: 30px;
      color: var(--border);
      font-size: 12px;
      letter-spacing: 2px;
      border-top: 1px solid var(--border);
    }
  </style>
</head>
<body>

<header>
  <p class="hero-label">AI — CSC 309 Project</p>
  <h1>Series Recommender</h1>
  <p class="subtitle">Collaborative Filtering · Cosine Similarity · User-Based Recommendations</p>
</header>

<main>

  <!-- ── USER SELECTION ── -->
  <p class="section-title">01 — Select a User</p>
  <div class="user-grid" id="user-grid"></div>

  <button id="recommend-btn" disabled>GET RECOMMENDATIONS</button>

  <!-- ── RESULTS ── -->
  <div id="results">
    <div class="notice">
      <span>▶</span>
      Select a user above and click the button to see their recommendations
    </div>
  </div>

  <!-- ── RATINGS TABLE ── -->
  <p class="section-title">02 — Full Ratings Table</p>
  <div class="table-wrap" id="ratings-table-wrap"></div>

  <!-- ── SIMILARITY TABLE ── -->
  <p class="section-title">03 — User Similarity Matrix</p>
  <div class="table-wrap" id="similarity-table-wrap"></div>

</main>

<footer>SERIES RECOMMENDATION SYSTEM &nbsp;·&nbsp; COLLABORATIVE FILTERING &nbsp;·&nbsp; CSC 309</footer>

<script>
  // ─────────────────────────────────────────────────────────
  // THE DATA — Same ratings table as the Python version
  // Rows = Users, Columns = Series
  // 1-5 = rating, 0 = not watched yet
  // ─────────────────────────────────────────────────────────

  const users = ['Ebube', 'Jude', 'Emma', 'Mark', 'Chikum', 'Adaeze'];

  const series = [
    'Peaky Blinders', 'Lucifer', 'Breaking Bad',
    'Game of Thrones', 'The Sandman', 'Stranger Things',
    'The Witcher', 'Money Heist', 'Squid Game', 'The Boys'
  ];

  // Each row = one user's ratings for all series (in the same order as the series array above)
  const ratingsMatrix = [
    [5, 4, 5, 0, 0, 3, 4, 0, 3, 0],  // Ebube
    [4, 0, 5, 4, 0, 5, 3, 4, 0, 5],  // Jude
    [0, 5, 3, 0, 5, 4, 0, 5, 4, 3],  // Emma
    [5, 3, 0, 5, 4, 0, 5, 0, 4, 0],  // Mark
    [0, 4, 0, 3, 5, 3, 0, 4, 5, 4],  // Chikum
    [3, 0, 4, 5, 0, 0, 4, 5, 3, 5],  // Adaeze
  ];


  // ─────────────────────────────────────────────────────────
  // COSINE SIMILARITY FUNCTION
  // Measures how similar two users are based on their ratings
  // Returns a value between 0 (different) and 1 (identical taste)
  // ─────────────────────────────────────────────────────────

  function cosineSimilarity(vecA, vecB) {
    let dotProduct = 0;
    let magnitudeA = 0;
    let magnitudeB = 0;

    for (let i = 0; i < vecA.length; i++) {
      dotProduct  += vecA[i] * vecB[i];
      magnitudeA  += vecA[i] * vecA[i];
      magnitudeB  += vecB[i] * vecB[i];
    }

    magnitudeA = Math.sqrt(magnitudeA);
    magnitudeB = Math.sqrt(magnitudeB);

    if (magnitudeA === 0 || magnitudeB === 0) return 0;
    return dotProduct / (magnitudeA * magnitudeB);
  }


  // ─────────────────────────────────────────────────────────
  // BUILD THE FULL SIMILARITY TABLE (all users vs all users)
  // ─────────────────────────────────────────────────────────

  function buildSimilarityMatrix() {
    const n = users.length;
    const matrix = Array.from({ length: n }, () => new Array(n).fill(0));

    for (let i = 0; i < n; i++) {
      for (let j = 0; j < n; j++) {
        matrix[i][j] = cosineSimilarity(ratingsMatrix[i], ratingsMatrix[j]);
      }
    }
    return matrix;
  }

  const similarityMatrix = buildSimilarityMatrix();


  // ─────────────────────────────────────────────────────────
  // RECOMMENDATION ENGINE
  // For a given user, find unwatched series and predict scores
  // ─────────────────────────────────────────────────────────

  function getRecommendations(targetIndex, topN = 3) {
    const targetRatings = ratingsMatrix[targetIndex];
    const recommendations = [];

    // Go through each series
    for (let s = 0; s < series.length; s++) {
      // Only consider series the target user has NOT watched
      if (targetRatings[s] !== 0) continue;

      let weightedSum = 0;
      let similaritySum = 0;

      // Look at every other user
      for (let u = 0; u < users.length; u++) {
        if (u === targetIndex) continue;

        const sim    = similarityMatrix[targetIndex][u];
        const rating = ratingsMatrix[u][s];

        // Only use users who watched this series AND are similar
        if (rating > 0 && sim > 0) {
          weightedSum   += sim * rating;
          similaritySum += sim;
        }
      }

      // Calculate the predicted score
      if (similaritySum > 0) {
        recommendations.push({
          name:  series[s],
          score: weightedSum / similaritySum
        });
      }
    }

    // Sort by score highest to lowest and return top N
    recommendations.sort((a, b) => b.score - a.score);
    return recommendations.slice(0, topN);
  }


  // ─────────────────────────────────────────────────────────
  // BUILD THE RATINGS TABLE ON THE PAGE
  // ─────────────────────────────────────────────────────────

  function buildRatingsTable() {
    const wrap = document.getElementById('ratings-table-wrap');
    let html = '<table><thead><tr><th>User</th>';
    series.forEach(s => { html += `<th>${s}</th>`; });
    html += '</tr></thead><tbody>';

    users.forEach((user, i) => {
      html += `<tr><td>${user}</td>`;
      ratingsMatrix[i].forEach(r => {
        const cls = r === 0 ? 'r0' : `r${r}`;
        const display = r === 0 ? '—' : r;
        html += `<td><span class="rating-cell ${cls}">${display}</span></td>`;
      });
      html += '</tr>';
    });

    html += '</tbody></table>';
    wrap.innerHTML = html;
  }


  // ─────────────────────────────────────────────────────────
  // BUILD THE SIMILARITY TABLE ON THE PAGE
  // ─────────────────────────────────────────────────────────

  function buildSimilarityTable() {
    const wrap = document.getElementById('similarity-table-wrap');
    let html = '<table><thead><tr><th>User</th>';
    users.forEach(u => { html += `<th>${u}</th>`; });
    html += '</tr></thead><tbody>';

    users.forEach((user, i) => {
      html += `<tr><td>${user}</td>`;
      users.forEach((_, j) => {
        const val = similarityMatrix[i][j];
        let cls = 'sim-low';
        if (val >= 0.75) cls = 'sim-high';
        else if (val >= 0.4) cls = 'sim-mid';
        const display = i === j ? '—' : val.toFixed(2);
        html += `<td class="sim-cell ${cls}">${display}</td>`;
      });
      html += '</tr>';
    });

    html += '</tbody></table>';
    wrap.innerHTML = html;
  }


  // ─────────────────────────────────────────────────────────
  // RENDER RECOMMENDATION CARDS
  // ─────────────────────────────────────────────────────────

  function renderRecommendations(userIndex) {
    const recs  = getRecommendations(userIndex, 3);
    const uName = users[userIndex];
    const resultsDiv = document.getElementById('results');

    if (recs.length === 0) {
      resultsDiv.innerHTML = `
        <div class="notice">
          <span>✓</span>
          ${uName} has already watched everything or there's not enough data.
        </div>`;
      return;
    }

    let html = `
      <div class="result-header">
        <div class="result-name">${uName}</div>
        <p>Top ${recs.length} recommended series based on similar users</p>
      </div>
      <div class="cards-grid">`;

    recs.forEach((rec, idx) => {
      const pct = ((rec.score / 5) * 100).toFixed(1);
      html += `
        <div class="rec-card">
          <div class="rank-badge">${String(idx + 1).padStart(2, '0')}</div>
          <div class="series-name">${rec.name}</div>
          <div class="score-bar-wrap">
            <div class="score-label">
              <span>Predicted Score</span>
              <span class="score-value">${rec.score.toFixed(2)} / 5.00</span>
            </div>
            <div class="score-bar">
              <div class="score-fill" data-width="${pct}"></div>
            </div>
          </div>
        </div>`;
    });

    html += '</div>';
    resultsDiv.innerHTML = html;

    // Animate the score bars after render
    setTimeout(() => {
      document.querySelectorAll('.score-fill').forEach(el => {
        el.style.width = el.dataset.width + '%';
      });
    }, 50);
  }


  // ─────────────────────────────────────────────────────────
  // SET UP USER BUTTONS
  // ─────────────────────────────────────────────────────────

  let selectedUser = null;

  function buildUserButtons() {
    const grid = document.getElementById('user-grid');
    users.forEach((name, i) => {
      const btn = document.createElement('button');
      btn.className = 'user-btn';
      btn.textContent = name;
      btn.addEventListener('click', () => {
        document.querySelectorAll('.user-btn').forEach(b => b.classList.remove('active'));
        btn.classList.add('active');
        selectedUser = i;
        document.getElementById('recommend-btn').disabled = false;
      });
      grid.appendChild(btn);
    });
  }

  document.getElementById('recommend-btn').addEventListener('click', () => {
    if (selectedUser !== null) renderRecommendations(selectedUser);
  });


  // ─────────────────────────────────────────────────────────
  // INITIALISE EVERYTHING WHEN PAGE LOADS
  // ─────────────────────────────────────────────────────────

  buildUserButtons();
  buildRatingsTable();
  buildSimilarityTable();

</script>
</body>
</html>

SyntaxError: invalid decimal literal (3148565060.py, line 27)